# 3.4 The perceptron algorithm

A linear separator learns by moving toward misclassified positives and away from misclassified negatives.

The perceptron algorithm uses empirical risk, validation behavior, and a cost-aware decision score. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))
    return rungs

def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)

def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


## The concept, built once on D1

The lesson formula is $$ w\leftarrow w+y_i x_i\quad\text{when }y_i(w^\top x_i)\le 0 $$. The next cell recomputes the exact loss average, cost, gap, and stabilized score from the plan.

In [ ]:

def the_perceptron_algorithm_method():
    losses = np.array([0.224, 0.07, 0.437], dtype=float)
    raw_sum = float(losses.sum())
    empirical_risk = round(float(raw_sum / len(losses)), 3)
    cost = 0.090
    score = round(empirical_risk + cost, 3)
    alternative = 0.386
    gap = round(alternative - score, 3)
    relative_gap = round(gap / alternative, 3)
    stable_score = round(0.80 * score, 3)
    final_score = min(score, alternative, stable_score)
    return {
        "losses": losses,
        "sum": raw_sum,
        "risk": empirical_risk,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stable": stable_score,
        "final": final_score,
    }

lesson_check = the_perceptron_algorithm_method()
print("losses:", lesson_check["losses"])
print("R_S =", round(lesson_check["sum"], 3), "/ 3 =", round(lesson_check["risk"], 3))
print("score =", round(lesson_check["score"], 3))
print("gap =", round(lesson_check["gap"], 3))
print("relative gap =", round(lesson_check["relative_gap"], 3))
print("stable score =", round(lesson_check["stable"], 3))
assert np.isclose(round(lesson_check["sum"], 3), 0.731)
assert np.isclose(round(lesson_check["risk"], 3), 0.244)
assert np.isclose(round(lesson_check["score"], 3), 0.334)
assert np.isclose(round(lesson_check["gap"], 3), 0.052)
assert np.isclose(round(lesson_check["relative_gap"], 3), 0.135)
assert np.isclose(round(lesson_check["stable"], 3), 0.267)


The assertions above keep the notebook and lesson prose on the same algorithmic scale.

In [ ]:

def safe_stratify(y):
    values, counts = np.unique(y, return_counts=True)
    if len(values) < 2:
        return None
    if counts.min() < 2:
        return None
    return y

def plot_2d_projection(ax, X, y, title):
    x_plot = X[:, :2]
    ax.scatter(x_plot[:, 0], x_plot[:, 1], c=y, cmap="viridis", s=16, alpha=0.75)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

def logistic_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    scaler = StandardScaler()
    x_tr_s = scaler.fit_transform(x_tr)
    x_te_s = scaler.transform(x_te)
    candidates = []
    for c_value in [0.05, 0.2, 1.0, 5.0]:
        model = LogisticRegression(C=c_value, max_iter=2000)
        model.fit(x_tr_s, y_tr)
        tr_prob = model.predict_proba(x_tr_s)
        te_prob = model.predict_proba(x_te_s)
        tr_pred = model.predict(x_tr_s)
        te_pred = model.predict(x_te_s)
        labels = model.classes_
        train_loss = log_loss(y_tr, tr_prob, labels=labels)
        val_loss = log_loss(y_te, te_prob, labels=labels)
        cost = 0.02 / c_value
        candidates.append({
            "C": c_value,
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "gap": float(val_loss - train_loss),
            "accuracy": float(accuracy_score(y_te, te_pred)),
            "cost": float(cost),
            "score": float(val_loss + cost),
            "pred": te_pred,
        })
    raw_winner = min(candidates, key=lambda item: item["val_loss"])
    fixed_winner = min(candidates, key=lambda item: item["score"])
    return candidates, raw_winner, fixed_winner

def run_logistic_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates, raw_winner, fixed_winner = logistic_candidates_for_rung(X, y)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": fixed_winner["val_loss"],
            "gap": fixed_winner["gap"],
            "accuracy": fixed_winner["accuracy"],
            "C": fixed_winner["C"],
            "score": fixed_winner["score"],
            "raw_C": raw_winner["C"],
        })
    return rows

def bias_variance_candidates_for_rung(X, y):
    stratify = safe_stratify(y)
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
    models = [
        ("linear", make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))),
        ("flexible-knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))),
    ]
    rows = []
    for label, model in models:
        model.fit(x_tr, y_tr)
        train_error = 1.0 - accuracy_score(y_tr, model.predict(x_tr))
        val_error = 1.0 - accuracy_score(y_te, model.predict(x_te))
        variance_proxy = abs(val_error - train_error)
        complexity_cost = 0.01 if label == "linear" else 0.04
        rows.append({
            "label": label,
            "train_error": float(train_error),
            "val_error": float(val_error),
            "variance_proxy": float(variance_proxy),
            "score": float(val_error + variance_proxy + complexity_cost),
        })
    return rows

def run_bias_variance_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        candidates = bias_variance_candidates_for_rung(X, y)
        winner = min(candidates, key=lambda item: item["score"])
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": winner["val_error"],
            "gap": winner["variance_proxy"],
            "model": winner["label"],
            "score": winner["score"],
        })
    return rows

def add_intercept(X):
    ones = np.ones((X.shape[0], 1))
    return np.hstack([ones, X])

def train_binary_perceptron(X, y_signed, epochs=60):
    X_aug = add_intercept(X)
    weights = np.zeros(X_aug.shape[1])
    mistakes = []
    for epoch in range(epochs):
        errors = 0
        for xi, yi in zip(X_aug, y_signed):
            margin = yi * float(np.dot(weights, xi))
            if margin <= 0:
                weights = weights + yi * xi
                errors = errors + 1
        mistakes.append(errors)
        if errors == 0:
            break
    return weights, mistakes

def train_ovr_perceptron(X, y, epochs=60):
    classes = np.unique(y)
    weights = []
    histories = []
    for cls in classes:
        y_signed = np.where(y == cls, 1, -1)
        w, hist = train_binary_perceptron(X, y_signed, epochs=epochs)
        weights.append(w)
        histories.append(hist)
    return classes, np.vstack(weights), histories

def predict_ovr_perceptron(classes, weights, X):
    scores = add_intercept(X).dot(weights.T)
    return classes[np.argmax(scores, axis=1)]

def perceptron_predictor(x_tr, y_tr, x_te):
    classes, weights, histories = train_ovr_perceptron(x_tr, y_tr, epochs=60)
    return predict_ovr_perceptron(classes, weights, x_te)

def run_perceptron_ladder():
    rows = []
    for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
        stratify = safe_stratify(y)
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=stratify)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        classes, weights, histories = train_ovr_perceptron(x_tr_s, y_tr, epochs=60)
        pred = predict_ovr_perceptron(classes, weights, x_te_s)
        rows.append({
            "rung": rung,
            "name": name,
            "n": X.shape[0],
            "d": X.shape[1],
            "classes": len(np.unique(y)),
            "metric": float(accuracy_score(y_te, pred)),
            "history": histories,
        })
    return rows


## The dataset ladder

D1 is inspectable by hand; D5 is a real 30-dimensional breast-cancer classification problem.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    values, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, :min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  class counts:", dict(zip(values.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


## Run the same method across D1–D5

The metric follows the plan: validation loss and generalization gap for 3.1–3.3, accuracy for 3.4.

In [ ]:

results = run_perceptron_ladder()
print("rung | accuracy | dimensions | classes")
for row in results:
    print(f"D{row['rung']} | {row['metric']:.3f} | {row['d']} | {row['classes']}")
helper_acc = clf_accuracy(perceptron_predictor, rungs[-1][1], rungs[-1][2])
print("D5 helper clf_accuracy(perceptron_predictor):", round(helper_acc, 3))


## Results visualization

The closing figure has one panel per rung plus a summary curve over D1–D5.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
flat_axes = axes.ravel()
for ax, (name, X, y), row in zip(flat_axes[:5], rungs, results):
    plot_2d_projection(ax, X, y, f"D{row['rung']} accuracy={row['metric']:.2f}")
flat_axes[5].plot([row["rung"] for row in results], [row["metric"] for row in results], marker="o", label="accuracy")
if "perceptron" != "perceptron":
    flat_axes[5].plot([row["rung"] for row in results], [abs(row["gap"]) for row in results], marker="s", label="gap")
flat_axes[5].set_xlabel("rung")
flat_axes[5].set_ylabel("accuracy")
flat_axes[5].legend()
fig.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

The hardest rung demonstrates why the raw term alone is not the decision rule.

In [ ]:

d5_name, d5_X, d5_y = rungs[-1]
stratify = safe_stratify(d5_y)
x_tr, x_te, y_tr, y_te = train_test_split(d5_X, d5_y, test_size=0.4, random_state=0, stratify=stratify)
classes, raw_weights, raw_histories = train_ovr_perceptron(x_tr, y_tr, epochs=60)
raw_pred = predict_ovr_perceptron(classes, raw_weights, x_te)
raw_acc = accuracy_score(y_te, raw_pred)
scaler = StandardScaler()
x_tr_s = scaler.fit_transform(x_tr)
x_te_s = scaler.transform(x_te)
classes, fixed_weights, fixed_histories = train_ovr_perceptron(x_tr_s, y_tr, epochs=60)
fixed_pred = predict_ovr_perceptron(classes, fixed_weights, x_te_s)
fixed_acc = accuracy_score(y_te, fixed_pred)
print("D5:", d5_name)
print("wrong raw-scale accuracy:", round(raw_acc, 3))
print("fixed scaled accuracy:", round(fixed_acc, 3))
print("wrong final mistakes:", raw_histories[0][-1])
print("fixed final mistakes:", fixed_histories[0][-1])
assert fixed_acc >= raw_acc


## Evaluate it + Practice

- Compare the metric with a no-skill baseline or `logistic_baseline`.
- Sanity check: shuffle labels and confirm the score degrades.
- Ablation: turn off the cost or scaling fix and watch the D5 choice or metric change.
- Failure signals: a large validation gap, a scale mismatch, or a raw-only winner.

Practice 1: Change one cost and rerun the D5 selection.

Practice 2: Repeat the D5 split with a different seed and compare the gap.

Practice 3: For skewed classes, add macro-F1 and compare it with accuracy.